# 01 — Visual Embeddings (BLIP-2)

Génère pour chaque photo de `data/processed/ALBUM/` :
- Une **caption** textuelle (BLIP-2 ou OpenCLIP fallback)
- Un **embedding 768-dim** (Q-Former de BLIP-2, espace vision-langage aligné)
- Les **métadonnées EXIF** : date, heure, GPS (si disponibles)

**Modèle** : `Salesforce/blip2-opt-2.7b` (float16 sur GPU, float32 sur CPU)  
**Cache modèle** : `data/models/` (téléchargé une seule fois, persisté sur l'hôte)  
**Input** : `data/processed/ALBUM/*.jpg`  
**Output** : `data/warehouse/memory_album/photo_embeddings/` (Parquet Snappy)

> Exception justifiée aux règles no-UDF : `mapInPandas` est nécessaire car
> le modèle PyTorch doit être chargé en Python pur. Il est chargé **une seule fois
> par partition**, pas par ligne.

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)

import yaml
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, FloatType, StringType, BooleanType,
    StructType, StructField, TimestampType
)

from config import (
    build_spark_session,
    ALBUM_PHOTOS_DIR, MEMORY_ALBUM_DIR, MODELS_CACHE_DIR, PROJECT_ROOT
)

OUTPUT_DIR = os.path.join(MEMORY_ALBUM_DIR, "photo_embeddings")

print(f"ALBUM_PHOTOS_DIR : {ALBUM_PHOTOS_DIR}")
print(f"OUTPUT_DIR       : {OUTPUT_DIR}")
print(f"MODELS_CACHE_DIR : {MODELS_CACHE_DIR}")

ALBUM_PHOTOS_DIR : /opt/spark/data/processed/ALBUM
OUTPUT_DIR       : /opt/spark/data/warehouse/memory_album/photo_embeddings
MODELS_CACHE_DIR : /opt/spark/data/models


In [2]:
# ── 1. PARAMÈTRES (config.yaml) ───────────────────────────────────────────────
_cfg_path = os.path.join(_d, 'config.yaml')
with open(_cfg_path, encoding='utf-8') as _f:
    _cfg = yaml.safe_load(_f)

_ma = _cfg.get('memory_album', {})
BLIP2_MODEL  = _ma.get('model', 'Salesforce/blip2-opt-2.7b')
BATCH_SIZE   = int(_ma.get('batch_size', 8))
N_PARTITIONS = int(_ma.get('n_partitions', 1))

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

print(f"Modèle        : {BLIP2_MODEL}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"N_PARTITIONS  : {N_PARTITIONS}")

Modèle        : Salesforce/blip2-opt-2.7b
Batch size    : 8
N_PARTITIONS  : 1


In [3]:
# ── 2. DÉCOUVERTE PHOTOS + EXTRACTION EXIF ────────────────────────────────────
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def _dms_to_decimal(dms, ref: str) -> float | None:
    """Convertit les coordonnées GPS DMS (degrés/minutes/secondes) en décimal."""
    if not dms or not ref:
        return None
    try:
        d, m, s = [float(v) for v in dms]
        dec = d + m / 60 + s / 3600
        return -dec if ref in ('S', 'W') else dec
    except Exception:
        return None

def extract_exif(path: str) -> dict:
    """Retourne {'exif_date': datetime|None, 'lat': float|None, 'lon': float|None}."""
    result = {'exif_date': None, 'lat': None, 'lon': None}
    try:
        img  = Image.open(path)
        raw  = img._getexif()
        if not raw:
            return result
        named = {TAGS.get(k, k): v for k, v in raw.items()}

        # Date
        date_str = named.get('DateTimeOriginal') or named.get('DateTime')
        if date_str:
            try:
                result['exif_date'] = datetime.strptime(date_str, '%Y:%m:%d %H:%M:%S')
            except ValueError:
                pass

        # GPS
        gps_raw = named.get('GPSInfo')
        if gps_raw and isinstance(gps_raw, dict):
            gps = {GPSTAGS.get(k, k): v for k, v in gps_raw.items()}
            result['lat'] = _dms_to_decimal(gps.get('GPSLatitude'),  gps.get('GPSLatitudeRef'))
            result['lon'] = _dms_to_decimal(gps.get('GPSLongitude'), gps.get('GPSLongitudeRef'))
    except Exception:
        pass
    return result

# Scan du dossier ALBUM
photo_rows = []
for p in sorted(Path(ALBUM_PHOTOS_DIR).rglob('*')):
    if p.suffix.lower() not in IMAGE_EXTS:
        continue
    exif = extract_exif(str(p))
    photo_rows.append({
        'photo_id'  : hashlib.md5(str(p).encode()).hexdigest()[:12],
        'path'      : str(p),
        'filename'  : p.name,
        'exif_date' : exif['exif_date'],
        'lat'       : exif['lat'],
        'lon'       : exif['lon'],
    })

print(f"{len(photo_rows)} photos trouvées dans {ALBUM_PHOTOS_DIR}")
n_with_date = sum(1 for r in photo_rows if r['exif_date'])
n_with_gps  = sum(1 for r in photo_rows if r['lat'] is not None)
print(f"  avec date EXIF : {n_with_date}")
print(f"  avec GPS       : {n_with_gps}")

64 photos trouvées dans /opt/spark/data/processed/ALBUM
  avec date EXIF : 49
  avec GPS       : 20


In [4]:
# ── 3. SESSION SPARK ──────────────────────────────────────────────────────────
from config import build_spark_session

spark = build_spark_session(
    'MyDigitalTwin-MemoryAlbum-Embeddings',
    delta=False,
    snappy=True,
    driver_memory='4g',
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} — prêt")

Spark 3.5.5 — prêt


26/05/01 18:17:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
# ── 4. SCHEMA + FONCTION mapInPandas ──────────────────────────────────────────

# Schema d'entrée (créé depuis photo_rows)
INPUT_SCHEMA = StructType([
    StructField('photo_id',  StringType(),    nullable=False),
    StructField('path',      StringType(),    nullable=False),
    StructField('filename',  StringType(),    nullable=False),
    StructField('exif_date', TimestampType(), nullable=True),
    StructField('lat',       FloatType(),     nullable=True),
    StructField('lon',       FloatType(),     nullable=True),
])

# Schema de sortie avec caption + embedding
OUTPUT_SCHEMA = StructType([
    StructField('photo_id',  StringType(),           nullable=False),
    StructField('path',      StringType(),           nullable=False),
    StructField('filename',  StringType(),           nullable=False),
    StructField('exif_date', TimestampType(),        nullable=True),
    StructField('lat',       FloatType(),            nullable=True),
    StructField('lon',       FloatType(),            nullable=True),
    StructField('caption',   StringType(),           nullable=True),
    StructField('embedding', ArrayType(FloatType()), nullable=True),
    StructField('has_gps',   BooleanType(),          nullable=False),
    StructField('model_used',StringType(),           nullable=False),
])

# Passés aux workers via broadcast
_BLIP2_MODEL_BC    = spark.sparkContext.broadcast(BLIP2_MODEL)
_CACHE_DIR_BC      = spark.sparkContext.broadcast(MODELS_CACHE_DIR)
_BATCH_SIZE_BC     = spark.sparkContext.broadcast(BATCH_SIZE)

def embed_partition(iterator):
    """
    mapInPandas : chargement du modèle UNE FOIS par partition.
    Stratégie :
      - GPU disponible  → BLIP-2 float16 (caption + embedding Q-Former 768-dim)
      - CPU uniquement  → OpenCLIP ViT-H-14 (embedding 1024-dim, pas de caption)
    """
    import torch
    import numpy as np
    from PIL import Image

    blip2_model = _BLIP2_MODEL_BC.value
    cache_dir   = _CACHE_DIR_BC.value
    batch_size  = _BATCH_SIZE_BC.value

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype  = torch.float16 if device == 'cuda' else torch.float32
    use_blip2 = (device == 'cuda')  # BLIP-2 en CPU est très lent pour les embeddings

    os.makedirs(cache_dir, exist_ok=True)

    if use_blip2:
        # ── BLIP-2 (GPU) : captions + embeddings Q-Former 768-dim ────────────
        from transformers import Blip2Processor, Blip2ForConditionalGeneration
        processor = Blip2Processor.from_pretrained(blip2_model, cache_dir=cache_dir)
        model = Blip2ForConditionalGeneration.from_pretrained(
            blip2_model, torch_dtype=dtype, cache_dir=cache_dir
        ).to(device)
        model.eval()
        model_name_used = blip2_model
    else:
        # ── OpenCLIP (CPU fallback) : embeddings 1024-dim, pas de caption ────
        import open_clip
        clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
            'ViT-H-14', pretrained='laion2b_s32b_b79k', cache_dir=cache_dir
        )
        clip_model.eval()
        model_name_used = 'openclip-ViT-H-14'

    for pdf in iterator:
        results = []

        for i in range(0, len(pdf), batch_size):
            batch = pdf.iloc[i : i + batch_size]
            images, valid_idx = [], []

            for j, row in enumerate(batch.itertuples()):
                try:
                    img = Image.open(row.path).convert('RGB')
                    images.append(img)
                    valid_idx.append(j)
                except Exception:
                    pass

            if not images:
                continue

            valid_rows = [batch.iloc[j] for j in valid_idx]

            if use_blip2:
                # ── Captions ──────────────────────────────────────────────────
                inputs = processor(
                    images=images, return_tensors='pt'
                ).to(device, dtype)

                with torch.no_grad():
                    gen_ids = model.generate(
                        **inputs, max_new_tokens=50, num_beams=1
                    )
                    captions = processor.batch_decode(
                        gen_ids, skip_special_tokens=True
                    )
                    captions = [c.strip() for c in captions]

                # ── Embeddings : Q-Former → mean pool → 768-dim ───────────────
                with torch.no_grad():
                    vision_out   = model.vision_model(
                        pixel_values=inputs.pixel_values
                    )
                    image_embeds = vision_out.last_hidden_state  # [B, N, 1408]
                    attn_mask    = torch.ones(
                        image_embeds.shape[:2], device=device, dtype=torch.long
                    )
                    query_tokens = model.query_tokens.expand(
                        image_embeds.shape[0], -1, -1
                    )  # [B, 32, 768]
                    qf_out = model.qformer(
                        query_embeds=query_tokens,
                        encoder_hidden_states=image_embeds,
                        encoder_attention_mask=attn_mask,
                    )
                    # Mean pool sur les 32 query tokens → [B, 768]
                    embeddings_t = qf_out.last_hidden_state.mean(dim=1)
                    embeddings   = embeddings_t.cpu().float().numpy()

                for k, row in enumerate(valid_rows):
                    results.append({
                        'photo_id'  : row['photo_id'],
                        'path'      : row['path'],
                        'filename'  : row['filename'],
                        'exif_date' : row['exif_date'],
                        'lat'       : float(row['lat']) if row['lat'] is not None and not np.isnan(float(row['lat'])) else None,
                        'lon'       : float(row['lon']) if row['lon'] is not None and not np.isnan(float(row['lon'])) else None,
                        'caption'   : captions[k],
                        'embedding' : embeddings[k].tolist(),
                        'has_gps'   : row['lat'] is not None,
                        'model_used': model_name_used,
                    })

            else:
                # ── OpenCLIP CPU fallback ─────────────────────────────────────
                import torch
                tensors = torch.stack([clip_preprocess(img) for img in images])
                with torch.no_grad():
                    embeddings = clip_model.encode_image(tensors)
                    embeddings = torch.nn.functional.normalize(embeddings, dim=-1)
                    embeddings = embeddings.cpu().float().numpy()

                for k, row in enumerate(valid_rows):
                    results.append({
                        'photo_id'  : row['photo_id'],
                        'path'      : row['path'],
                        'filename'  : row['filename'],
                        'exif_date' : row['exif_date'],
                        'lat'       : float(row['lat']) if row['lat'] is not None and not np.isnan(float(row['lat'])) else None,
                        'lon'       : float(row['lon']) if row['lon'] is not None and not np.isnan(float(row['lon'])) else None,
                        'caption'   : None,
                        'embedding' : embeddings[k].tolist(),
                        'has_gps'   : row['lat'] is not None,
                        'model_used': model_name_used,
                    })

        if results:
            yield pd.DataFrame(results)

In [6]:
# ── 5. PIPELINE SPARK ─────────────────────────────────────────────────────────
df_input = (
    spark
    .createDataFrame(photo_rows, schema=INPUT_SCHEMA)
    .repartition(N_PARTITIONS)
)
print(f"DataFrame input : {df_input.count()} photos, {N_PARTITIONS} partition(s)")

df_embeddings = df_input.mapInPandas(embed_partition, schema=OUTPUT_SCHEMA)

(
    df_embeddings
    .write
    .mode('overwrite')
    .option('compression', 'snappy')
    .parquet(OUTPUT_DIR)
)
print(f"✓ Embeddings écrits dans {OUTPUT_DIR}")

DataFrame input : 64 photos, 1 partition(s)


✓ Embeddings écrits dans /opt/spark/data/warehouse/memory_album/photo_embeddings


In [7]:
# ── 6. VALIDATION ─────────────────────────────────────────────────────────────
df_check = spark.read.parquet(OUTPUT_DIR)

total      = df_check.count()
with_cap   = df_check.filter(F.col('caption').isNotNull()).count()
with_embed = df_check.filter(F.col('embedding').isNotNull()).count()
with_gps   = df_check.filter(F.col('has_gps') == True).count()
with_date  = df_check.filter(F.col('exif_date').isNotNull()).count()
models     = df_check.select('model_used').distinct().collect()

print(f"\n{'─'*50}")
print(f"Photos traitées  : {total}")
print(f"Avec caption     : {with_cap} ({with_cap/total*100:.0f}%)")
print(f"Avec embedding   : {with_embed} ({with_embed/total*100:.0f}%)")
print(f"Avec GPS         : {with_gps} ({with_gps/total*100:.0f}%)")
print(f"Avec date EXIF   : {with_date} ({with_date/total*100:.0f}%)")
print(f"Modèle(s)        : {[r.model_used for r in models]}")
print(f"{'─'*50}")

print("\n── Exemples de captions ──────────────────────────")
df_check.select('filename', 'caption').filter(
    F.col('caption').isNotNull()
).show(5, truncate=80)

print("\n── IDs photos (pour les manual_overrides dans config.yaml) ──")
df_check.select('photo_id', 'filename', 'exif_date').orderBy('exif_date').show(20, truncate=50)


──────────────────────────────────────────────────
Photos traitées  : 64
Avec caption     : 0 (0%)
Avec embedding   : 64 (100%)
Avec GPS         : 64 (100%)
Avec date EXIF   : 49 (77%)
Modèle(s)        : ['openclip-ViT-H-14']
──────────────────────────────────────────────────

── Exemples de captions ──────────────────────────
+--------+-------+
|filename|caption|
+--------+-------+
+--------+-------+


── IDs photos (pour les manual_overrides dans config.yaml) ──
+------------+-----------------------------------------+-------------------+
|    photo_id|                                 filename|          exif_date|
+------------+-----------------------------------------+-------------------+
|7bc973043c61|06d6a51d-0fbf-4cb6-9ad4-e17cd73314ad.JPEG|               NULL|
|14f997c73707|126740D0-9D6C-4C8F-9800-92BC07EE2558.JPEG|               NULL|
|4192c11cbd6a|270FE8AB-76F9-40F8-AEB7-DCCED6986B38.JPEG|               NULL|
|1fd656bfaef3|3068686B-04AE-4D84-98D2-F1F8136BA72D.JPEG|            

In [8]:
spark.stop()